# Toy particle system

2D particle system of $N$ particles with exactlty $N/2$ species and two particles per species.

In [1]:
import diffrax as dfx
import distrax as dsx
import equinox as eqx
import jax
import jax.numpy as jnp

from superiorflows import DistributionDataSource, Flow
from superiorflows.train import (
    MaximumLikelihoodLoss,
)


class ToyParticles(eqx.Module):
    positions: jnp.ndarray
    species: jnp.ndarray
    box: jnp.ndarray

    @property
    def N(self):
        return self.positions.shape[-2]

    @classmethod
    def get_dynamic_mask(cls):
        return cls(positions=True, species=False, box=False)

/Users/Leonardo/Documents/Postdoc/Projects/superiorflows/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_species = 3
alphas = jnp.ones(n_species)
betas = jnp.ones(n_species)
L = 2.0

assert alphas.shape == betas.shape
N = 2 * len(alphas)
prior_dist = dsx.Uniform(low=jnp.zeros(2), high=L*jnp.ones(2))
angle_dist = dsx.Uniform(low=jnp.zeros(2), high=2*jnp.pi*jnp.ones(2))
norm_dist = dsx.Transformed(
    distribution=dsx.Beta(
        alpha=alphas,
        beta=betas
    ),
    bijector=dsx.ScalarAffine(
        shift=jnp.zeros((n_species,)),
        scale=(L / 2.0) * jnp.ones((n_species,))
    )
)

In [70]:
import distrax as dsx
import equinox as eqx
import jax.numpy as jnp


# --- 1. The Data Container (The Event) ---
class ToyParticles(eqx.Module):
    positions: jnp.ndarray  # Shape: (..., N, 2)
    species: jnp.ndarray    # Shape: (..., N)
    box: jnp.ndarray        # Shape: (..., 2)

    @property
    def N(self):
        return self.positions.shape[-2]

# --- 2. The Distribution ---
class ToyParticlesDistribution(eqx.Module, dsx.Distribution):
    L: float = eqx.field(static=True)
    alphas: jnp.ndarray
    betas: jnp.ndarray
    n_species: int = eqx.field(static=True)
    n_particles: int = eqx.field(static=True)

    def __init__(self, n_species, L, alphas=None, betas=None):
        self.n_species = n_species
        self.n_particles = 2 * n_species
        self.L = float(L)
        self.alphas = 2.0 * jnp.ones(n_species) if alphas is None else jnp.asarray(alphas)
        self.betas = 2.0 * jnp.ones(n_species) if betas is None else jnp.asarray(betas)

        if self.alphas.shape[0] != n_species:
             raise ValueError(f"Alphas length mismatch: {self.alphas.shape[0]} vs {n_species}")

    @property
    def event_shape(self):
        # Describes the shape of a SINGLE sample (unbatched)
        return ToyParticles(
            positions=(self.n_particles, 2),
            species=(self.n_particles,),
            box=(2,)
        )

    @property
    def _prior_dist(self):
        return dsx.Uniform(low=jnp.zeros(2), high=jnp.full(2, self.L))

    @property
    def _angle_dist(self):
        return dsx.Uniform(low=jnp.zeros(self.n_species), high=2*jnp.pi*jnp.ones(self.n_species))

    @property
    def _norm_dist(self):
        return dsx.Transformed(
            distribution=dsx.Beta(alpha=self.alphas, beta=self.betas),
            bijector=dsx.ScalarAffine(shift=jnp.zeros(self.n_species), scale=(self.L / 2.0))
        )

    def _sample_n(self, key, n):
        k1, k2, k3 = jax.random.split(key, 3)

        # 1. Sample Centers (Species 0, 1, 2...)
        # Shape: (n, n_species, 2)
        x_centers = self._prior_dist.sample(seed=k1, sample_shape=(n, self.n_species))

        # 2. Sample Relative Polar Coords
        # Shape: (n, n_species)
        angles = self._angle_dist.sample(seed=k2, sample_shape=(n,))
        radii = self._norm_dist.sample(seed=k3, sample_shape=(n,))

        # 3. Compute Satellites
        dx = radii * jnp.cos(angles)
        dy = radii * jnp.sin(angles)
        delta = jnp.stack([dx, dy], axis=-1)

        # Apply PBC: (Center + Delta) % L
        x_satellites = jnp.remainder(x_centers + delta, self.L)

        # 4. Interleave to Sort: [Center0, Sat0, Center1, Sat1...]
        # Stack: (n, n_species, 2, 2) -> (n, 2*n_species, 2)
        X_pairs = jnp.stack([x_centers, x_satellites], axis=2)
        positions = X_pairs.reshape(n, self.n_particles, 2)

        # 5. Create Species Labels
        # [0, 0, 1, 1, 2, 2...]
        s_single = jnp.repeat(jnp.arange(self.n_species), 2)
        species = jnp.broadcast_to(s_single, (n, self.n_particles))

        # 6. Box
        batched_box = jnp.full((n, 2), self.L)

        return ToyParticles(positions=positions, species=species, box=batched_box)

    def log_prob(self, value: ToyParticles):
        # 1. Parse Inputs (Handle potential batching implicitly via JAX logic)
        X = value.positions  # (..., N, 2)
        a = value.species    # (..., N)

        # 2. Sort X based on species `a` to ensure [C0, S0, C1, S1] ordering
        # This makes the logic robust even if input is shuffled
        idx_sort = jnp.argsort(a, axis=-1)
        X_sorted = jnp.take_along_axis(X, idx_sort[..., None], axis=-2)

        # 3. Reshape into Pairs: (..., n_species, 2_particles, 2_coords)
        # axis -2 is the pair index: 0=Center, 1=Satellite
        # We need to reshape the last dimension N -> (n_species, 2)
        # Note: We must be careful about the batch dims. We assume the last dim is N.
        batch_shape = X.shape[:-2]
        X_pairs = X_sorted.reshape(*batch_shape, self.n_species, 2, 2)

        x_centers = X_pairs[..., 0, :]    # (..., n_species, 2)
        x_satellites = X_pairs[..., 1, :] # (..., n_species, 2)

        # 4. Compute Log Probs

        # A. Prior (Centers)
        lp_prior = self._prior_dist.log_prob(x_centers).sum(axis=(-1, -2)) # Sum over dims and species

        # B. Geometric Relations (PBC difference)
        diff = x_satellites - x_centers
        diff = diff - jnp.round(diff / self.L) * self.L

        norm = jnp.linalg.norm(diff, axis=-1)       # (..., n_species)
        angle = jnp.arctan2(diff[..., 1], diff[..., 0]) # (..., n_species)
        angle = jnp.remainder(angle, 2 * jnp.pi)

        # C. Conditionals
        # Radius probability
        lp_norm = self._norm_dist.log_prob(norm).sum(axis=-1)

        # Angle probability
        lp_angle = self._angle_dist.log_prob(angle).sum(axis=-1)

        # Jacobian adjustment: -log(r)
        # Logic: P(x,y) = P(r,th) / r  => log P(x,y) = log P(r) + log P(th) - log(r)
        lp_jacobian = -jnp.sum(jnp.log(norm), axis=-1)

        # 5. Boundary Checks (Hard constraints)
        # Check if any radius exceeds L/2
        valid_norm = jnp.all(norm < (self.L / 2.0), axis=-1)

        total_lp = lp_prior + lp_norm + lp_angle + lp_jacobian

        return jnp.where(valid_norm, total_lp, -jnp.inf)

In [71]:
n_species = 3
alphas = jnp.ones(n_species)
betas = jnp.ones(n_species)
L = 2.0
distribution = ToyParticlesDistribution(n_species, L, alphas=alphas, betas=betas)

key = jax.random.key(0)
key, subkey = jax.random.split(key)
samples = distribution.sample(seed=key, sample_shape=4)
distribution.log_prob(samples)

Array([-8.425433 , -5.683769 , -6.4369297, -2.4959488], dtype=float32)

In [5]:
class UniformToyParticles(eqx.Module, dsx.Distribution):
    L: float = eqx.field(static=True)
    ref_species: jnp.ndarray  # Shape: (N,)

    def __init__(self, L, ref_species):
        self.L = float(L)
        self.ref_species = jnp.asarray(ref_species)

    @property
    def n_particles(self):
        return self.ref_species.shape[0]

    @property
    def event_shape(self):
        return ToyParticles(
            positions=(self.n_particles, 2),
            species=(self.n_particles,),
            box=(2,) # scalar broadcasted to 2 dims
        )

    def _sample_n(self, key, n):
        k_pos, k_spec = jax.random.split(key)

        # 1. Sample Positions: Uniform(0, L)
        # Shape: (n, N, 2)
        pos = jax.random.uniform(
            k_pos,
            shape=(n, self.n_particles, 2),
            minval=0.0,
            maxval=self.L
        )

        # 2. Sample Species: Random permutation of ref_species
        # We use vmap to permute each sample in the batch independently
        def _permute(k):
            return jax.random.permutation(k, self.ref_species)

        keys_perm = jax.random.split(k_spec, n)
        species = jax.vmap(_permute)(keys_perm)

        # 3. Box: Broadcast scalar L
        batched_box = jnp.full((n, 2), self.L)

        return ToyParticles(positions=pos, species=species, box=batched_box)

    def log_prob(self, value: ToyParticles):
        # 1. Base Log Prob: -N * log(Volume)
        # Volume = L^2
        # log(Volume) = 2 * log(L)
        base_log_prob = -self.n_particles * (2.0 * jnp.log(self.L))

        # 2. Check Constraints
        # A. Positions inside box [0, L]
        in_box = jnp.all(
            (value.positions >= 0.0) & (value.positions <= self.L),
            axis=(-1, -2)
        )

        # B. Correct Species Composition
        # Sort both to compare histograms (composition) rather than order
        sorted_val_species = jnp.sort(value.species, axis=-1)
        sorted_ref_species = jnp.sort(self.ref_species, axis=-1)

        # We use isclose in case of floats, though usually species are ints.
        # Equality is safer for ints.
        valid_composition = jnp.all(sorted_val_species == sorted_ref_species, axis=-1)

        # 3. Return
        is_valid = in_box & valid_composition
        return jnp.where(is_valid, base_log_prob, -jnp.inf)

In [6]:
# Create reference species (e.g. 3 of type 0, 3 of type 1)
ref_species = jnp.array([0, 0, 0, 1, 1, 1])

# Initialize
uniform_dist = UniformToyParticles(L=2.0, ref_species=ref_species)

# Sample
key = jax.random.key(42)
samples = uniform_dist.sample(seed=key, sample_shape=(10,))

# Check
print(samples.positions.shape) # (10, 6, 2)
print(samples.species[0])      # Shuffled version of ref_species

(10, 6, 2)
[1 1 1 0 0 0]


In [63]:
class ParticlesMLPVelocity(eqx.Module):
    mlp: eqx.nn.MLP
    N: int = eqx.field(static=True)
    d: int = eqx.field(static=True)

    def __init__(self, N: int, d: int, width: int, depth: int, *, key):
        self.N = N
        self.d = d
        self.mlp = eqx.nn.MLP(
            in_size=(d + 1) * N + 1, # positions + species + time
            out_size=N * d,
            width_size=width,
            depth=depth,
            activation=jax.nn.tanh,
            key=key,
        )

    @eqx.filter_jit
    def __call__(self, t, x, ctx):
        flatten_positions = x.positions.ravel()
        species_embedding = ctx.species.astype(jnp.float32)
        t_feat = jnp.array([t])
        velocity_flat = self.mlp(jnp.concatenate([flatten_positions, species_embedding, t_feat]))
        velocity = velocity_flat.reshape((self.N, self.d))
        return ToyParticles(
            positions=velocity,
            species=None,
            box=jnp.zeros_like(x.box)
        )



In [ ]:
cls

NameError: name 'cls' is not defined

In [66]:
vellocity_field = ParticlesMLPVelocity(N=2*n_species, d=2, width=16, depth=2, key=jax.random.key(0))
batch_size = 32
data_source = DistributionDataSource(distribution, batch_size)
flow_kwargs = dict(stepsize_controller=dfx.PIDController(rtol=1e-5, atol=1e-5))
loss_fn = MaximumLikelihoodLoss(base_distribution=uniform_dist, **flow_kwargs)

flow = Flow(base_distribution=uniform_dist, velocity_field=velocity_field, **flow_kwargs)

In [67]:
flow.sample_and_log_prob(seed=0, sample_shape=(2,))

ValueError: Terms are not compatible with solver! Got:
ODETerm(
  vector_field=partial(
    <function _augmented_dynamics>,
    random_vectors=None,
    velocity_field=ParticlesMLPVelocity(
      mlp=MLP(
        layers=(
          Linear(
            weight=f32[16,19](jax),
            bias=f32[16](jax),
            in_features=19,
            out_features=16,
            use_bias=True
          ),
          Linear(
            weight=f32[16,16](jax),
            bias=f32[16](jax),
            in_features=16,
            out_features=16,
            use_bias=True
          ),
          Linear(
            weight=f32[12,16](jax),
            bias=f32[12](jax),
            in_features=16,
            out_features=12,
            use_bias=True
          )
        ),
        activation=<PjitFunction of <function tanh at 0x111ba9c60>>,
        final_activation=<function <lambda>>,
        use_bias=True,
        use_final_bias=True,
        in_size=19,
        out_size=12,
        width_siz

In [55]:
y0, ctx = eqx.partition(samples, flow.dynamic_mask)

In [80]:
jnp.arange(n_species).repeat(2)

Array([0, 0, 1, 1, 2, 2], dtype=int32)

## Test script

In [85]:
import diffrax as dfx
import distrax as dsx
import equinox as eqx
import jax
import jax.numpy as jnp
import optax
from pathlib import Path
import importlib.util
import sys

from superiorflows import DistributionDataSource, Flow
from superiorflows.train import (
    CheckpointCallback,
    LoggerCallback,
    MaximumLikelihoodLoss,
    ProgressBarCallback,
    Trainer,
)

# Import module dynamically
script_path = Path.cwd().parent / "scripts/toy_particles.py"
spec = importlib.util.spec_from_file_location("toy_particles", script_path)
toy = importlib.util.module_from_spec(spec)
sys.modules["toy"] = toy
spec.loader.exec_module(toy)

In [86]:
key = jax.random.key(0)
n_species = 3
L = 2.0
alphas = jnp.ones(n_species)
betas = jnp.ones(n_species)
target_dist = toy.ToyParticlesDistribution(n_species, L, alphas=alphas, betas=betas)
ref_species = jnp.arange(n_species).repeat(2)
base_dist = toy.UniformToyParticles(L=L, ref_species=ref_species)

width = 16
depth = 2
key, subkey = jax.random.split(key)
velocity_field = toy.ParticlesMLPVelocity(N=2*n_species, d=2, width=width, depth=depth, key=subkey)
flow_kwargs = dict(
        stepsize_controller=dfx.PIDController(rtol=1e-5, atol=1e-5),
        dynamic_mask=toy.ToyParticles.get_dynamic_mask()
    )

flow = Flow(
    base_distribution=base_dist,
    velocity_field=velocity_field,
    **flow_kwargs,
    )


key, subkey = jax.random.split(key)
training_samples = target_dist.sample(seed=key, sample_shape=100)


In [87]:
key, subkey = jax.random.split(key)
flow.sample_and_log_prob(seed=subkey, sample_shape=100)

(ToyParticles(positions=f32[100,6,2], species=i32[100,6], box=weak_f32[100,2]),
 Array([-8.197599 , -8.43917  , -8.38833  , -8.404583 , -8.060577 ,
        -8.656635 , -7.9970865, -7.852998 , -8.406082 , -8.13323  ,
        -8.369085 , -8.325434 , -8.347672 , -8.011145 , -8.128102 ,
        -8.013735 , -8.304911 , -8.086618 , -8.395746 , -8.613187 ,
        -8.051994 , -8.487962 , -8.009833 , -8.9627   , -8.132686 ,
        -8.433689 , -8.286964 , -8.029414 , -7.9752703, -8.391392 ,
        -7.5986805, -7.6574206, -8.207962 , -8.604536 , -8.24844  ,
        -8.481996 , -8.215564 , -8.466976 , -8.420599 , -7.860776 ,
        -8.273685 , -8.2294445, -8.522114 , -8.8039   , -8.015151 ,
        -7.89858  , -8.1704645, -7.662086 , -8.321063 , -7.789608 ,
        -8.364465 , -8.271392 , -7.968993 , -8.098347 , -8.350577 ,
        -8.619229 , -8.492932 , -8.110421 , -8.4974375, -7.7302394,
        -8.416434 , -8.434048 , -8.2180805, -8.621974 , -8.541649 ,
        -8.324139 , -8.050295 , -8.3

In [88]:
flow.log_prob(training_samples)

Array([-8.321174 , -8.64324  , -8.413328 , -8.152027 , -8.6721735,
       -8.475405 , -8.149569 , -8.426302 , -8.475817 , -8.2207775,
       -8.276801 , -8.196962 , -8.196184 , -8.067212 , -8.550541 ,
       -8.800694 , -8.124864 , -8.173768 , -8.316549 , -8.279616 ,
       -8.417331 , -8.312221 , -8.554349 , -8.112169 , -8.4414215,
       -9.139999 , -8.614603 , -8.39013  , -7.8537574, -8.81128  ,
       -8.1480875, -8.34595  , -8.028732 , -8.355432 , -8.467114 ,
       -8.043751 , -8.254603 , -8.523588 , -8.09532  , -8.539006 ,
       -8.247274 , -8.60632  , -8.685643 , -8.852514 , -7.9731646,
       -8.23287  , -8.365999 , -8.362389 , -7.903818 , -9.027325 ,
       -7.9702973, -8.586863 , -8.459099 , -8.230483 , -8.363898 ,
       -8.291799 , -8.236958 , -8.152424 , -8.274076 , -8.458517 ,
       -8.464588 , -8.696165 , -8.413043 , -8.598242 , -8.12601  ,
       -8.730322 , -8.251024 , -8.208664 , -8.149803 , -9.203906 ,
       -8.8596115, -8.239226 , -8.289905 , -8.368066 , -8.0872

In [89]:
system = jax.tree_util.tree_map(lambda x: x[0], training_samples)
f = jnp.zeros(())
sol = flow.integrate_augmented_ode(system, t0=1.0, t1=0.0, logq0=f, key=None)
logq0 = sol.ys['logq']
x0 = sol.ys['x']

In [90]:
X, Logq = flow.sample_and_log_prob(seed=subkey, sample_shape=100)
X = eqx.tree_at(lambda x: x.positions, X, X.positions % X.box[:, None, :])

In [91]:
target_dist.log_prob(X)

Array([-9.104629 , -8.078723 ,        nan, -7.128331 , -8.684066 ,
       -7.8776035,        nan,        nan,        nan,        nan,
       -5.009037 , -8.539798 , -8.850774 , -8.608752 ,        nan,
              nan,        nan, -8.996196 ,        nan, -8.583017 ,
       -7.743998 , -8.103835 , -9.29121  , -7.738412 ,        nan,
              nan, -8.637788 ,        nan,        nan, -7.990528 ,
              nan,        nan,        nan,        nan, -8.000034 ,
       -8.77684  , -7.850341 ,        nan,        nan,        nan,
              nan, -7.849759 , -8.099446 , -8.73524  ,        nan,
              nan, -8.946586 , -9.153819 , -8.758903 , -8.726194 ,
              nan,        nan,        nan, -6.991475 ,        nan,
              nan, -6.4081173,        nan, -6.548493 , -9.068602 ,
       -7.5549726,        nan, -7.050905 , -7.0474267,        nan,
              nan, -6.498083 , -9.06366  , -7.9741526, -7.4896536,
              nan,        nan, -7.5943727,        nan,        

In [ ]:
def fold_back(x):
    return x

ToyParticles(positions=f32[100,6,2], species=i32[100,6], box=weak_f32[100,2])

In [74]:
X.box[0]

Array([2., 2.], dtype=float32, weak_type=True)

## Test actual particle system

In [1]:
import time
from pathlib import Path

import diffrax as dfx
import distrax as dsx
import equinox as eqx
import jax
import jax.numpy as jnp
import optax
import typer
from superiorflows import DistributionDataSource
from superiorflows.train import (
    CheckpointCallback,
    LoggerCallback,
    MaximumLikelihoodLoss,
    ProgressBarCallback,
    Trainer,
)
from typing_extensions import Annotated


class ParticleSystem(eqx.Module):
    positions: jnp.ndarray
    species: jnp.ndarray
    box: jnp.ndarray

    @property
    def d(self):
        return self.positions.shape[-1]

    @property
    def N(self):
        return self.positions.shape[-2]

    @classmethod
    def get_dynamic_mask(cls):
        return cls(positions=True, species=False, box=False)

class UniformParticles(eqx.Module, dsx.Distribution):
    N : int = eqx.field(static=True)
    d : int = eqx.field(static=True)
    L: float = eqx.field(static=True)
    ref_species: jnp.ndarray

    def __init__(self, N, d, L, composition):
        self.L = float(L)
        self.N = N
        self.d = d
        self.ref_species = jnp.concatenate([
            jnp.full(int(x * N), i, dtype=int)
            for i, x in enumerate(composition)
        ])
        assert sum(composition) == 1.0
        assert jnp.all(self.ref_species.shape[0] == N)

    @property
    def event_shape(self):
        return ParticleSystem(
            positions=(self.N, self.d),
            species=(self.N,),
            box=(self.d,), 
        )

    def _sample_n(self, key, n):
        k_pos, k_spec = jax.random.split(key)

        # 1. Sample Positions: Uniform(0, L)
        # Shape: (n, N, d)
        pos = jax.random.uniform(k_pos, shape=(n, self.N, self.d), minval=0.0, maxval=self.L)

        # 2. Sample Species: Random permutation of ref_species
        def _permute(k):
            return jax.random.permutation(k, self.ref_species)

        keys_perm = jax.random.split(k_spec, n)
        species = jax.vmap(_permute)(keys_perm)

        # 3. Box: Broadcast scalar L
        batched_box = jnp.full((n, self.d), self.L)

        return ParticleSystem(positions=pos, species=species, box=batched_box)

    def log_prob(self, value: ParticleSystem):
        # We assume configurations outside the box are valid 
        # as they represent configurations
        # that would be projected back into the box.

        # 1. Base Log Prob: -N * log(Volume)
        base_log_prob = -self.N * (self.d * jnp.log(self.L))

        # 2. Correct Species Composition
        sorted_val_species = jnp.sort(value.species, axis=-1)
        sorted_ref_species = jnp.sort(self.ref_species, axis=-1)
        valid_composition = jnp.all(sorted_val_species == sorted_ref_species, axis=-1)

        # 3. Return
        return jnp.where(valid_composition, base_log_prob, -jnp.inf)


/Users/Leonardo/Documents/Postdoc/Projects/superiorflows/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
N = 22
d = 2
L = 2.0
uniform_dist = UniformParticles(
    N=N,
    d=d,
    L=L,
    composition = (5/11, 3/11, 3/11)
)
key = jax.random.key(0)
key, subkey = jax.random.split(key)
samples = uniform_dist.sample(seed=subkey, sample_shape=(4,))

In [3]:
uniform_dist.log_prob(samples)

Array([-30.498476, -30.498476, -30.498476, -30.498476],      dtype=float32, weak_type=True)

In [76]:
class ParticlesMLPVelocity(eqx.Module):
    mlp: eqx.nn.MLP
    N: int = eqx.field(static=True)
    d: int = eqx.field(static=True)

    def __init__(self, N: int, d: int, width: int, depth: int, *, key):
        self.N = N
        self.d = d
        self.mlp = eqx.nn.MLP(
            in_size=(2 * d) * N + N + 1,  # positions (sin/cos) + species + time
            out_size=N * d,
            width_size=width,
            depth=depth,
            activation=jax.nn.tanh,
            key=key,
        )

    @eqx.filter_jit
    def __call__(self, t, x, ctx):
        # Embed positions: [sin(2pi*x/L), cos(2pi*x/L)]
        L = ctx.box[0]
        k = 2 * jnp.pi / L
        pos_embed = jnp.concatenate([jnp.sin(k * x.positions), jnp.cos(k * x.positions)], axis=-1)

        flatten_embedding = pos_embed.ravel()
        species_embedding = ctx.species.astype(jnp.float32)
        t_feat = jnp.broadcast_to(t, ctx.box.shape[:-1] + (1,))
        velocity_flat = self.mlp(jnp.concatenate([flatten_embedding, species_embedding, t_feat]))
        velocity = velocity_flat.reshape((self.N, self.d))

        return ParticleSystem(positions=velocity, species=None, box=None)

In [83]:
y0, ctx = eqx.partition(sample, ParticleSystem.get_dynamic_mask())
key, subkey = jax.random.split(key)
velocity_field = ParticlesMLPVelocity(N=N, d=d, width=16, depth=3, key=subkey)
velocity_field(0.5, y0, ctx)

ParticleSystem(positions=f32[22,2], species=None, box=None)

In [85]:
L = ctx.box[0]
k = 2 * jnp.pi / L
pos_embed = jnp.concatenate([jnp.sin(k * y0.positions), jnp.cos(k * y0.positions)], axis=-1)
flatten_embedding = pos_embed.ravel()
species_embedding = ctx.species.astype(jnp.float32)

In [88]:
species_embedding.shape

(22,)

In [34]:
import atooms.postprocessing as pp
from atooms.trajectory import TrajectoryXYZ

path = '/Users/Leonardo/Documents/PhD/Projects/ParticlesMC/data/datasets/JBB25/T1.0/N44/M100/steps1000000/seed42/trajectories/2/trajectory.xyz'
trj = TrajectoryXYZ(path)

In [35]:
print(trj[-1])

system composed of N=44 particles
with chemical composition C={'1': 20, '2': 12, '3': 12}
with chemical concentration x={'1': 0.45454545454545453, '2': 0.2727272727272727, '3': 0.2727272727272727}
enclosed in a cubic box at number density rho=1.192075



In [36]:
len(trj)

100

In [37]:
trj.metadata

{'npart': 44,
 'step': 10000,
 'columns': ['species', 'position'],
 'dt': 1,
 'cell': [6.075395792383285, 6.075395792383285],
 'rho': 1.1920748468939728,
 'T': 1.0,
 'model': 'JBB',
 'potential_energy_per_particle': -1.9271769162133692,
 'ndim': 2}

In [38]:
trj[-1].particle

[Particle(species=1, mass=1.0, position=[2.624498 2.490387], velocity=[0. 0. 0.], radius=0.5),
 Particle(species=1, mass=1.0, position=[-4.044768  5.215544], velocity=[0. 0. 0.], radius=0.5),
 Particle(species=2, mass=1.0, position=[10.497127 16.551246], velocity=[0. 0. 0.], radius=0.5),
 Particle(species=2, mass=1.0, position=[-6.881447 -9.438642], velocity=[0. 0. 0.], radius=0.5),
 Particle(species=1, mass=1.0, position=[ -5.522579 -10.422229], velocity=[0. 0. 0.], radius=0.5),
 Particle(species=1, mass=1.0, position=[ 3.476598 -4.515219], velocity=[0. 0. 0.], radius=0.5),
 Particle(species=1, mass=1.0, position=[-0.084434  5.269603], velocity=[0. 0. 0.], radius=0.5),
 Particle(species=1, mass=1.0, position=[13.516479  0.0719  ], velocity=[0. 0. 0.], radius=0.5),
 Particle(species=2, mass=1.0, position=[-9.502751  1.61558 ], velocity=[0. 0. 0.], radius=0.5),
 Particle(species=3, mass=1.0, position=[-4.478263  3.854751], velocity=[0. 0. 0.], radius=0.5),
 Particle(species=3, mass=1.0,

In [41]:
trj[-1].view('position')

array([[  2.624498,   2.490387],
       [ -4.044768,   5.215544],
       [ 10.497127,  16.551246],
       [ -6.881447,  -9.438642],
       [ -5.522579, -10.422229],
       [  3.476598,  -4.515219],
       [ -0.084434,   5.269603],
       [ 13.516479,   0.0719  ],
       [ -9.502751,   1.61558 ],
       [ -4.478263,   3.854751],
       [ 10.54814 ,  18.28188 ],
       [-14.225749,   5.126913],
       [ 28.724334,   2.454145],
       [ -9.644154,   4.295287],
       [  2.988492,  11.06106 ],
       [  2.893477,   6.75499 ],
       [  6.845514,   5.498163],
       [ -1.585912,  13.762886],
       [ 11.076797,   0.786756],
       [-15.620826,   9.468648],
       [ 10.013303,   0.842768],
       [ -6.270739,   7.133147],
       [ 10.660916,  21.749788],
       [  0.900957,  -3.486416],
       [ 40.046904,   6.051216],
       [  8.778468,   5.774916],
       [ 13.759462,   8.090089],
       [ -2.510293,  14.548597],
       [ -5.292168,   9.64888 ],
       [ -2.588776,   9.362657],
       [-1

In [46]:
x = [1, 2, 3]
y = [4, 5]
tp = (x, y)
x[1] = 20
x
tp

([1, 20, 3], [4, 5])

## Test the actual code

In [1]:
import sys
from pathlib import Path
import jax.numpy as jnp

sys.path.append(str(Path().resolve().parent))

from particle_systems.particle_system import ParticleSystem, UniformParticles, BoltzmannDistribution, TrajectoryDataSource


# Single file
path = '/Users/Leonardo/Documents/PhD/Projects/ParticlesMC/data/datasets/JBB25/T1.0/N44/M100/steps1000000/seed42/trajectories/2/trajectory.xyz'
data_source = TrajectoryDataSource(path)

In [2]:
data_source[2]

ParticleSystem(
  positions=f32[44,2](numpy), species=i32[44](numpy), box=f32[2](numpy)
)

In [4]:
N = data_source.metadata['npart']
d = data_source.metadata['ndim']
L = data_source.metadata['cell'][0]
temperature = data_source.metadata['T']
composition = (5/11, 3/11, 3/11)

JBB = {
    "potential": [
        {
            "type": "lennard_jones",
            "parameters": {
                "exponent": 12,
                "sigma": [[1.0, 0.8, 0.9], [0.8, 0.88, 0.8], [0.9, 0.8, 0.94]],
                "epsilon": [[1.0, 1.5, 0.75], [1.5, 0.5, 1.5], [0.75, 1.5, 0.75]],
            },
        }
    ],
    "cutoff": [
        {
            "type": "smooth",
            "parameters": {"rcut": [[2.5, 2.0, 2.25], [2.0, 2.2, 2.0], [2.25, 2.0, 2.35]]},
        }
    ],
}

target_dist = BoltzmannDistribution(N=N, d=d, L=L, temperature=temperature, model=JBB, composition=composition)
-target_dist.log_prob(data_source[2]) * temperature / N


Array(-2.2527325, dtype=float32)

In [4]:
# All trajectories in a directory (AWESOME)
path = '/Users/Leonardo/Documents/PhD/Projects/ParticlesMC/data/datasets/JBB25/T1.0/N44/M100/'
data_source = TrajectoryDataSource(path)
len(data_source)

110000

In [5]:
data_loader = data_source.to_dataset(batch_size=64, shuffle=False)
data_loader[1718] == data_loader[0]

True

In [9]:
N = data_source.metadata['npart']
d = data_source.metadata['ndim']
L = data_source.metadata['cell'][0]
temperature = data_source.metadata['T']
composition = (5/11, 3/11, 3/11)

JBB = {
    "potential": [
        {
            "type": "lennard_jones",
            "parameters": {
                "exponent": 12,
                "sigma": [[1.0, 0.8, 0.9], [0.8, 0.88, 0.8], [0.9, 0.8, 0.94]],
                "epsilon": [[1.0, 1.5, 0.75], [1.5, 0.5, 1.5], [0.75, 1.5, 0.75]],
            },
        }
    ],
    "cutoff": [
        {
            "type": "quadratic_cut_shift",
            "parameters": {"rcut": [[2.5, 2.0, 2.25], [2.0, 2.2, 2.0], [2.25, 2.0, 2.35]]},
        }
    ],
}

target_dist = BoltzmannDistribution(N=N, d=d, L=L, temperature=temperature, model=JBB, composition=composition)

In [10]:
data_loader = data_source.to_dataset(batch_size=64, shuffle=False)
samples = data_loader[4]
samples

ParticleSystem(
  positions=f32[64,44,2](numpy), species=i32[64,44](numpy), box=f32[64,2](numpy)
)

In [11]:
target_dist.log_prob(samples)

Array([76.74299 , 89.77191 , 77.43506 , 84.29615 , 78.77132 , 83.60739 ,
       77.44466 , 69.07558 , 74.82376 , 75.94481 , 75.818504, 63.657177,
       71.78237 , 68.142426, 70.13699 , 79.05056 , 85.142815, 75.85566 ,
       80.80509 , 77.766785, 78.31918 , 73.561455, 71.68047 , 78.43595 ,
       73.72218 , 77.73509 , 72.64883 , 62.042824, 76.12432 , 69.91967 ,
       79.609535, 75.71158 , 74.26602 , 76.16656 , 92.97921 , 75.56045 ,
       65.23337 , 80.60437 , 72.91171 , 78.91454 , 84.89409 , 70.681175,
       67.856   , 71.74054 , 78.29039 , 75.83201 , 76.71172 , 72.88469 ,
       83.88118 , 68.3618  , 75.95512 , 77.699295, 71.844536, 75.22638 ,
       86.01921 , 79.0119  , 69.94076 , 75.32692 , 80.9491  , 83.70983 ,
       62.40839 , 72.00437 , 78.20254 , 74.87513 ], dtype=float32)

In [12]:
target_dist.log_prob(data_source[0])

Array(61.460705, dtype=float32)

In [13]:
# IPL
path = '/Users/Leonardo/Documents/PhD/Projects/ParticlesMC/data/datasets/SS142D/T0.1/N44/M100/steps1000000/seed42/trajectories/2/trajectory.xyz'
data_source = TrajectoryDataSource(path)
N = data_source.metadata['npart']
d = data_source.metadata['ndim']
L = data_source.metadata['cell'][0]
temperature = data_source.metadata['T']
composition = (1/2, 1/2)
model = {
    "potential": [
        {
            "type": "inverse_power",
            "parameters": {
                "exponent": 12,
                "sigma": [[1.0, 1.2], [1.2, 1.4]],
                "epsilon": [[1.0, 1.0], [1.0, 1.0]],
            },
        }
    ],
    "cutoff": [
        {
            "type": "cut_shift",
            "parameters": {"rcut": [[2.5, 3.0], [3.0, 3.5]]},
        }
    ],
}

target_dist = BoltzmannDistribution(N=N, d=d, L=L, temperature=temperature, model=model, composition=composition)

In [14]:
system = data_source[0] # true potential energy per particle: 0.34191972408857796
-target_dist.log_prob(system) * temperature / N

Array(0.34191963, dtype=float32)